# 06e hybrid head: the sequence model plus four raw numbers about the judged transaction

Step 6 extension, the experiment 6a's error analysis points at. The eight frauds only XGBoost caught were isolated large amounts (median amt_z 7.3, median amount 840 dollars); the 20 only the transformer caught were small amounts inside a burst. The AMT token has 20 bins whose top bin starts at the train 99th percentile, so 600 and 6,000 dollars are the same token: the sequence model cannot see how large "large" is. XGBoost reads the raw amount and its z-score against the card's history.

**What is trained, and why.** The reference recipe (06b H12: full fine-tuning from `pretrain/best`, last-token pooling, lr 5e-5, 3 epochs, normals sampled to 5%, three seeds) with exactly one change: the head receives the last token's 384-vector concatenated with four standardised numbers about the judged row, and is two layers instead of one.

| part | parameters | trained | source |
|---|---|---|---|
| GPT-2 body (wte, wpe, 6 blocks, ln_f) | 11,061,888 | yes, lr 5e-5 | `pretrain/best` |
| head: Linear(384 + 4, 64), GELU, Linear(64, 2) | 25,026 | yes, new | random init |
| features: log_amt, amt_z, amt_ratio_mean, log gap_prev_s | 0 | no | `baseline.build_features`, standardised on train rows only, NaN (no history) = 0 |

Three checks: the usual 3-seed table against the reference in units of the reference sd; recall by amt_z bucket for reference, hybrid and XGBoost (does the hybrid close the large-amount gap without losing the burst frauds); and the 6a disagreement rows themselves (how many of the 8 XGBoost-only frauds the hybrid now flags, how many of the 20 transformer-only it keeps). Same machine as the reference; resumable; about 1.2 h on Apple MPS.

In [ ]:
import os, sys, glob, shutil
try:
    from google.colab import drive, userdata                      # Colab: mount Drive, pull the repo, sync new notebooks to Drive
    drive.mount("/content/drive")
    tok = userdata.get("GH_TOKEN")
    if not os.path.exists("/content/transaction-fm"):
        !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
    else:
        !cd /content/transaction-fm && git pull -q
    %pip install -q datasets xgboost
    %cd /content/transaction-fm
except ImportError:                                               # laptop: this notebook is the repo copy; data under $TFM_ROOT
    here = os.getcwd()
    while not os.path.isdir(os.path.join(here, "src", "tfm")):
        assert here != os.path.dirname(here), "run this notebook from inside the transaction-fm repo"
        here = os.path.dirname(here)
    %cd {here}
sys.path.insert(0, os.path.abspath("src"))
from tfm.runtime import root, describe, on_colab
ROOT = root()
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
if on_colab():
    for f in glob.glob("notebooks/*.ipynb"):                      # new notebooks from the repo appear in Drive
        dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok |", describe(), "| ROOT =", ROOT)

## Stage 6e-0: data, the H12 examples, the four features, the reference

In [ ]:
import json, time, os, numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
from pathlib import Path
from tfm.tokenizer import TransactionTokenizer
from tfm.finetune import build_examples
from tfm.baseline import build_features
from tfm.metrics import evaluate_scores, mean_sd, recall_at_alert_rate
from tfm.ablate import finetune_run, trainable_summary, load_model
from tfm.hybrid import HYBRID_FEATURES, numeric_features, hybrid_spec, head_size
from tfm.runtime import accelerator

FT_OVERRIDES = {}                                   # empty in a real run; the smoke run sets tiny values here
P, PRE, RES = f"{ROOT}/data/processed", f"{ROOT}/pretrain/best", f"{ROOT}/results"
os.makedirs(RES, exist_ok=True)
df = pd.read_parquet(f"{P}/transactions.parquet")
tok = TransactionTokenizer.load(f"{PRE}/tokenizer.json")
ids_all = tok.encode(df)
card, split, fraud = df["card_id"].to_numpy(), df["split"].to_numpy(), df["is_fraud"].to_numpy()
PAD, BOS, V = tok.vocab["<PAD>"], tok.vocab["<BOS>"], len(tok.vocab)
ALERT, SEEDS = 0.006, (0, 1, 2)
KEYS = ["pr_auc", "roc_auc", "recall_at_alert", "card_discovery", "fp_ratio"]
DEV = accelerator()
t0 = time.time(); feats = build_features(df); F, stats = numeric_features(df, feats)
print(f"features for {len(df):,} rows in {time.time() - t0:.1f} s; standardisation fit on train rows:")
display(pd.DataFrame(stats).T.style.format("{:.3f}"))
show = df[df["split"] == "test"].index[:6]
display(pd.concat([df.loc[show, ["card_id", "ts", "amt", "is_fraud"]], feats.loc[show, HYBRID_FEATURES].round(3).add_prefix("raw "), pd.DataFrame(F[show], index=show, columns=[f"std {c}" for c in HYBRID_FEATURES]).round(2)], axis=1))
examples = {("val", 12): build_examples(ids_all, card, split, fraud, "val", 12, PAD, BOS), ("test", 12): build_examples(ids_all, card, split, fraud, "test", 12, PAD, BOS)}
Xt, yt, it = examples[("test", 12)]
def read_json(name):
    for p in (f"{RES}/{name}.json", f"results/metrics/{name}.json"):
        if os.path.exists(p): return json.load(open(p))
    raise FileNotFoundError(name)
REF = [read_json(f"06b_H12_last_seed{k}") for k in SEEDS]
REF_SCORES = [np.load(f"{ROOT}/ablate/H12_last/seed{k}/test_scores.npy") for k in SEEDS]
ref = mean_sd([r["test"] for r in REF]); REF_SD = ref["pr_auc"]["sd"]
here = describe()
same = all(r["runtime"].get("machine") == here["machine"] and r["runtime"].get("accelerator") == here["accelerator"] for r in REF)
print("accelerator:", DEV, "| reference 06b H12 last:", f"{ref['pr_auc']['mean']:.4f} +- {REF_SD:.4f}", "| same machine as the reference:", same)

## Stage 6e-1: what is trained, counted on the real model

In [ ]:
m = load_model("pretrained", "last", PRE, V, PAD, BOS, arch="hybrid", n_feats=F.shape[1])
counts = trainable_summary(m); del m
print({k: f"{v:,}" for k, v in counts.items()})
print(f"head = Linear({384 + F.shape[1]}, 64) + GELU + Linear(64, 2) = {head_size(384, F.shape[1]):,} parameters; reference head was 384 x 2 = 768")

## Stage 6e-2: three seeds

In [ ]:
t0 = time.time()
runs = [finetune_run(hybrid_spec(k), ids_all, card, split, fraud, PAD, BOS, V, PRE, ROOT, RES, examples, keep=0.05, alert=ALERT, overrides=FT_OVERRIDES, feats_all=F) for k in SEEDS]
hyb = mean_sd([r["test"] for r in runs])
HYB_SCORES = [np.load(f"{ROOT}/extras/hybrid/seed{k}/test_scores.npy") for k in SEEDS]
print(f"{(time.time() - t0) / 60:.1f} min this session")
tab = pd.DataFrame([{"arm": "reference: last-token head (06b H12)", "trainable": REF[0].get("trainable_params", "all"), **{k: f"{ref[k]['mean']:.4f} +- {ref[k]['sd']:.4f}" for k in KEYS}, "PR-AUC - ref": 0.0, "in ref sd": 0.0},
                    {"arm": "hybrid: last token + 4 raw features", "trainable": runs[0]["trainable_params"], **{k: f"{hyb[k]['mean']:.4f} +- {hyb[k]['sd']:.4f}" for k in KEYS},
                     "PR-AUC - ref": hyb["pr_auc"]["mean"] - ref["pr_auc"]["mean"], "in ref sd": (hyb["pr_auc"]["mean"] - ref["pr_auc"]["mean"]) / max(REF_SD, 1e-9)}])
display(tab.style.format({"PR-AUC - ref": "{:+.4f}", "in ref sd": "{:+.1f}"}).hide(axis="index"))

## Stage 6e-3: recall by amount z-score, three models

Frauds in the test rows are bucketed by amt_z (the judged amount against the card's own history, from the step-5 features). At the fixed 0.6% alert budget, recall per bucket for the reference (seed 0), the hybrid (seed 0) and XGBoost full (seed 0, from step 5). The large-amount buckets are where 6a found XGBoost ahead; the small-amount burst frauds are where the transformer was.

In [ ]:
import matplotlib.pyplot as plt
def flagged(score): f = np.zeros(len(score), bool); f[np.argsort(-score, kind="stable")[:max(1, int(round(ALERT * len(score))))]] = True; return f
z = feats["amt_z"].to_numpy()[it]                       # the judged row's amount z-score, aligned with the test examples
edges = [-np.inf, 0, 2, 5, 10, np.inf]; labels = ["<0", "0-2", "2-5", "5-10", "10+", "no history"]
bucket = np.where(np.isnan(z), 5, np.digitize(z, edges[1:-1]))
models = {"transformer (06b, seed 0)": REF_SCORES[0], "hybrid (seed 0)": HYB_SCORES[0]}
xgb_path = f"{ROOT}/baseline/full/seed0/test_scores.npy"
if os.path.exists(xgb_path):
    xs = np.load(xgb_path)
    if len(xs) == len(yt): models["XGBoost 29 features (05, seed 0)"] = xs
    else: print("baseline scores have a different row count, XGBoost line skipped")
fr = yt == 1
rows = []
for b, lab in enumerate(labels):
    m = fr & (bucket == b)
    if m.sum() == 0: continue
    row = {"amt_z bucket": lab, "n fraud": int(m.sum())}
    for name, s in models.items():
        row[f"recall {name}"] = float(flagged(s)[m].mean())
    rows.append(row)
zt = pd.DataFrame(rows); display(zt.style.format({c: "{:.3f}" for c in zt.columns if c.startswith("recall")}).hide(axis="index"))
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(zt)); w = 0.8 / len(models)
for i, name in enumerate(models):
    ax.bar(x + i * w - 0.4 + w / 2, zt[f"recall {name}"], w, label=name)
ax.set_xticks(x); ax.set_xticklabels([f"{a}\n(n={b})" for a, b in zip(zt["amt_z bucket"], zt["n fraud"])]); ax.set_ylabel(f"recall at {ALERT:.1%} alerts"); ax.set_xlabel("judged amount, z-score vs the card's history")
ax.set_ylim(0, 1.02); ax.legend(loc="lower left", fontsize=8); ax.set_title("where each model catches fraud")
plt.tight_layout(); fig.savefig(f"{RES}/06e_amt_z_recall.png", dpi=120); plt.show()

## Stage 6e-4: the 6a disagreement rows

`06a_validate.json` lists the test rows of the frauds only the transformer flagged (20) and only XGBoost flagged (8), both at their own 0.6% cutoffs on seed 0. The same rows, scored by the hybrid seed 0: how many of the XGBoost-only frauds it now flags, and how many of the transformer-only ones it keeps.

In [ ]:
try:
    dis = read_json("06a_validate")["disagreements"]
    only_tf, only_xgb = np.array(dis["transformer_only"], int), np.array(dis["xgboost_only"], int)
    assert only_tf.max(initial=-1) < len(yt) and only_xgb.max(initial=-1) < len(yt)
    f_ref, f_hyb = flagged(REF_SCORES[0]), flagged(HYB_SCORES[0])
    dtab = pd.DataFrame([{"rows": "XGBoost-only frauds (6a)", "n": len(only_xgb), "flagged by reference seed 0": int(f_ref[only_xgb].sum()), "flagged by hybrid seed 0": int(f_hyb[only_xgb].sum()),
                          "median hybrid score": float(np.median(HYB_SCORES[0][only_xgb])) if len(only_xgb) else float("nan"), "median reference score": float(np.median(REF_SCORES[0][only_xgb])) if len(only_xgb) else float("nan")},
                         {"rows": "transformer-only frauds (6a)", "n": len(only_tf), "flagged by reference seed 0": int(f_ref[only_tf].sum()), "flagged by hybrid seed 0": int(f_hyb[only_tf].sum()),
                          "median hybrid score": float(np.median(HYB_SCORES[0][only_tf])) if len(only_tf) else float("nan"), "median reference score": float(np.median(REF_SCORES[0][only_tf])) if len(only_tf) else float("nan")}])
    display(dtab.style.format({"median hybrid score": "{:.4f}", "median reference score": "{:.4f}"}).hide(axis="index"))
    disagreement = dtab.to_dict(orient="records")
except (FileNotFoundError, AssertionError) as e:
    print("6a disagreement rows not available for this data:", repr(e)); disagreement = None

## Save 06e_hybrid.json

In [ ]:
out = {"reference": {"runs": [r["name"] for r in REF], "aggregate": ref, "runtime": REF[0]["runtime"]}, "features": HYBRID_FEATURES, "standardisation": stats,
       "head_params": counts["score"], "total_params": counts["total"], "runs": runs, "aggregate": hyb,
       "pr_auc_minus_ref": hyb["pr_auc"]["mean"] - ref["pr_auc"]["mean"], "in_ref_sd": (hyb["pr_auc"]["mean"] - ref["pr_auc"]["mean"]) / max(REF_SD, 1e-9),
       "recall_by_amt_z": zt.to_dict(orient="records"), "disagreement_rows": disagreement, "runtime": describe()}
with open(f"{RES}/06e_hybrid.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print("saved 06e_hybrid.json:", list(out))

## What to read off the numbers

- The 3-seed table: a gain under 2 reference sd is a tie. Even a tie is informative if the amt_z table moves: the hybrid may trade burst frauds for large-amount frauds at the same budget.
- The amt_z buckets: the 10+ bucket is where XGBoost was ahead in 6a; if the hybrid matches XGBoost there and keeps the reference's recall in the <0 and 0-2 buckets (the bursts), the tokenizer's amount cap was the limitation, and the fix is cheap (four numbers into the head) rather than a new tokenizer.
- The disagreement rows: eight and twenty rows are too few for a statistic; they are the qualitative check that the mechanism is the one 6a described.
- For the report: this is the one place the project combines the two model families; if it helps, the recommendation is not "transformer or XGBoost" but "a sequence representation plus the few raw numbers that binning destroys".

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
if on_colab():
    def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
        cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
        return cells(a) == cells(b)
    for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
        repo = f"notebooks/{os.path.basename(f)}"
        if not os.path.exists(repo) or same_code(f, repo):
            shutil.copy(f, repo)
        else:
            print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
    !git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
    !git add results notebooks && git commit -q -m "results and notebooks from Colab" && git pull --rebase -q && git push -q   # the other 06c group may have pushed meanwhile; its files are disjoint
    !git log --oneline | head -1
else:
    print("laptop: results copied into results/; this notebook file already is the repo copy; commit from the terminal")
    !git status --short results notebooks

In [ ]:
if on_colab():
    drive.flush_and_unmount()